# Review the PII Replacement Plan Before Training

#### What you'll learn

Before training, NeMo Safe Synthesizer replaces personally identifiable information (PII) in your data. An LLM drafts the replacement plan: which columns hold PII, what kind of PII each one holds, and how the new values should be generated. In this notebook, you'll draft that plan without training anything, review it, edit it if needed, and then train with exactly the plan you reviewed.

The steps are:

1. Draft the plan from the data with `plan_pii_replacement()`.
2. Review the saved plan file, `pii_replacement_plan.yaml`.
3. Edit the file if the plan is wrong or incomplete.
4. Run the pipeline with the reviewed file and check the PII results in the evaluation report.

> Runtime: Running the complete notebook takes about an hour on an NVIDIA A100, most of it fine-tuning. A first run may take longer while model files are downloaded.

> The dataset used here is the simulated hospital admissions dataset from the [Healthcare Hospital Readmissions](healthcare-hospital-readmissions.ipynb) tutorial. It was generated with [NVIDIA NeMo Data Designer](https://docs.nvidia.com/nemo/datadesigner/getting-started/welcome) and contains no real patient data.

## 1. Environment

Run the cell below to install NeMo Safe Synthesizer with the engine and CUDA 12.9 extras. If it is already installed in your notebook environment, you can skip this cell. A GPU that supports CUDA 12.9 is required for planning, training, and generation. The cell requires [uv](https://docs.astral.sh/uv/).

In [ ]:
# SPDX-FileCopyrightText: Copyright (c) 2025-2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
# SPDX-License-Identifier: Apache-2.0

import sys

# The override resolves a flashinfer-python pin conflict between vLLM and NeMo Safe Synthesizer.
!echo "flashinfer-python==0.6.16.post4" | uv pip install --python "{sys.executable}" \
    "nemo-safe-synthesizer[engine,cu129]==0.1.14" \
    -c https://raw.githubusercontent.com/NVIDIA-NeMo/Safe-Synthesizer/v0.1.14/constraints.txt \
    --overrides - \
    --find-links https://flashinfer.ai/whl/cu129/flashinfer-jit-cache/ \
    --find-links https://flashinfer.ai/whl/flashinfer-cubin/ \
    --find-links https://wheels.vllm.ai/0.27.0/cu129/vllm/ \
    --find-links https://download.pytorch.org/whl/cu129/torch/ \
    --find-links https://download.pytorch.org/whl/cu129/torchaudio/ \
    --find-links https://download.pytorch.org/whl/cu129/torchvision/ \
    --find-links https://download.pytorch.org/whl/cu129/torchcodec/ \
    --find-links https://download.pytorch.org/whl/cu129/torchao/ \
    --find-links https://download.pytorch.org/whl/cu129/triton/

In [ ]:
import base64

import pandas as pd
from IPython.display import Code, IFrame, Markdown, display

## 2. Load the admissions data

Pandas infers quoted numeric CSV fields as integers, so the explicit string dtype below preserves leading zeros in postal codes.

In [ ]:
source_df = pd.read_csv(
    "datasets/hospital_readmissions_synthetic_2026_08_18.csv",
    dtype={"postal_code": "string"},
)

print(f"Loaded {len(source_df):,} admissions with {source_df.shape[1]} fields.")
display(source_df.head(3))

## 3. Choose where PII planning runs

By default, the planning LLM runs in a temporary vLLM server on this machine's GPU, so no API key is needed and the data stays on this machine. The local model needs an 80 GB GPU and downloads about 66 GB the first time it runs.

To use a hosted service instead, such as the NVIDIA API at `https://integrate.api.nvidia.com/v1`, set `NSS_INFERENCE_ENDPOINT`, `NSS_INFERENCE_MODEL`, and `NSS_INFERENCE_KEY` in the next cell. You can obtain an API key from [build.nvidia.com](https://build.nvidia.com/settings/api-keys). Column samples are then sent to that service. See [Where planning requests go](../product-overview/pii_replacement.md#where-planning-requests-go) for details.

In [ ]:
import os

# Keep the data split and generated replacements reproducible across tutorial runs.
SPLIT_SEED = 318

# Optional: plan with a hosted service instead of a local vLLM server.
# import getpass
# os.environ["NSS_INFERENCE_ENDPOINT"] = "https://integrate.api.nvidia.com/v1"
# os.environ["NSS_INFERENCE_MODEL"] = "nvidia/nemotron-3-ultra-550b-a55b"
# os.environ["NSS_INFERENCE_KEY"] = getpass.getpass("Paste inference API key: ")
if os.environ.get("NSS_INFERENCE_ENDPOINT"):
    print(f"PII planning sends requests to {os.environ['NSS_INFERENCE_ENDPOINT']}")
else:
    print("PII planning runs a local vLLM server on this machine's GPU")

## 4. Draft the plan

`plan_pii_replacement()` asks the planning LLM which columns contain PII and how to replace them. Nothing is replaced or trained yet. With `output_path`, it saves the complete configuration and, beside it, the part to review: `pii_replacement_plan.yaml`. Run this cell once; the next cell shows the file.

In [ ]:
from nemo_safe_synthesizer.sdk.library_builder import SafeSynthesizer

ARTIFACT_ROOT = "safe-synthesizer-review-artifacts"
builder = (
    SafeSynthesizer(save_path=ARTIFACT_ROOT)
    .with_data_source(source_df)
    .with_replace_pii(seed=SPLIT_SEED)
    .with_data(random_state=SPLIT_SEED)
    .with_generate(num_records=5_000)
)

RUN_DIR = builder._workdir.run_dir
builder.plan_pii_replacement(output_path=RUN_DIR / "safe-synthesizer-config.yaml")
PLAN_PATH = RUN_DIR / "pii_replacement_plan.yaml"
print(f"Draft plan saved to {PLAN_PATH}")

In [ ]:
# Show the plan file. Rerun this cell after editing the file; it does not draft the plan again.
display(Code(PLAN_PATH.read_text(), language="yaml"))

## 5. Review the plan

The plan is a draft from an LLM, so check it against what you know about the data. The file has two sections, each with a short comment above it:

- `columns_to_replace` lists the columns whose values are replaced. Every column not listed is kept unchanged.
- `data_to_sampler_value_mapping` translates values of the columns that guide a replacement, such as `sex`, to values the sampler understands. NSS fills it in.

The header names the sampler and locale the plan was written for. Those are set in the configuration as `replace_pii.sampler` and `replace_pii.locale`, not in this file.

Check the plan:

- **Nothing sensitive is kept.** Every identifier, name, contact detail, street address, and date of birth should be in `columns_to_replace`. A sensitive column left out reaches the model as is.
- **Nothing is replaced that you need to keep.** Clinical fields such as diagnosis, labs, and charges should not be in the plan.
- **Each `entity_type` is right.** It decides what kind of value is generated, for example `full_name` or `unique_identifier`.
- **Each `pattern` keeps the format you need.** An optional `pattern` sets the format of the new values. Without one, an identifier gets random letters and digits in the original's layout, so a fixed prefix such as `MRN-` is not kept.
- **The `depends_on` links make sense.** They list the columns a replacement follows, for example an email built from the replaced name, or a name chosen to match `sex`. A missing link means those values are replaced independently, so they may no longer match.
- **Free text is scanned.** Written notes such as `discharge_summary` should be listed with `entity_type: free_text`, so names and other PII inside them are found and replaced.

Some kinds of values are never replaced as whole columns: cities, states, ZIP codes, countries, organizations, genders, ethnicities, and ordinary dates such as admission dates. They stay out of the plan by design, and the plan may use them to guide other replacements.

## 6. Edit the plan

To change the plan, edit `pii_replacement_plan.yaml` at the path the draft cell printed, then rerun the display cell above to see the result.

- **Columns:** add, remove, or correct entries in `columns_to_replace`. For example, to keep the `MRN-` prefix of medical record numbers such as `MRN-4C806C7840`, give the `mrn` entry a pattern. `&` stands for a digit or an uppercase letter, and `#` for a digit:

  ```yaml
  - column_name: mrn
    entity_type: unique_identifier
    pattern: MRN-&&&&&&&&&&
  ```

- **Value mapping:** in `data_to_sampler_value_mapping`, map each value as it appears in your data to one or more values the sampler accepts. Gender uses `female` and `male` for every sampler. Matching ignores case, and a value mapped to several sampler values can draw from any of them. For example:

  ```yaml
  data_to_sampler_value_mapping:
    sex:
      W: [female]
      Unknown: [female, male]
  ```

When the plan is right, run the next cell. It loads and checks the file. A sampler value with a typo stops with a message naming it. A value in a guiding column that matches no sampler value and has no entry also stops, with a message giving the column and how many values are missing.

In [ ]:
# Use the reviewed file. Its plan is no longer a draft, so the planning LLM does not run again.
builder = builder.with_replace_pii(replacement_plan=str(PLAN_PATH), seed=SPLIT_SEED)
reviewed = builder.plan_pii_replacement().inline_plan
print(f"Using the reviewed plan: {len(reviewed.columns_to_replace)} columns will be replaced.")

## 7. Train, generate, and evaluate

The next cell runs the complete pipeline with the reviewed plan. It sets aside 499 records for evaluation, replaces PII in the remaining 9,498 records before fine-tuning the default SmolLM3 model, generates 5,000 synthetic records, and evaluates the result. The outputs are saved under `ARTIFACT_ROOT`.

In [ ]:
builder.run()
results = builder.results
print(f"Generated {len(results.synthetic_data):,} synthetic admissions.")

## 8. Check the PII results in the evaluation report

Open the **Data Privacy** page of the report below. Its **PII Replacement and Replay** table lists each replaced column, how its values were replaced, how much PII was found in the training data, and how many original values reappear in the synthetic data (PII Replay). Free-text columns list each kind of PII found in them. These results support review; they do not certify compliance or approve the data for sharing.

In [ ]:
report_html = results.evaluation_report_html
if report_html:
    report_url = "data:text/html;base64," + base64.b64encode(report_html.encode("utf-8")).decode("ascii")
    display(IFrame(src=report_url, width="100%", height=850))
else:
    display(Markdown("The run did not return an HTML evaluation report."))

## Next steps

- To reuse a reviewed plan in a later run, pass its path with `with_replace_pii(replacement_plan=...)`, or set `replace_pii.replacement_plan` in a configuration file.
- See the [PII replacement guide](../product-overview/pii_replacement.md) for every entity type, pattern symbol, and setting.
- See the [Healthcare Hospital Readmissions](healthcare-hospital-readmissions.ipynb) tutorial for utility checks on the same data.